In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================
# 9-MODEL ENSEMBLE (RAM-SAFE, DRIVE-CHECKPOINTED)
# Libraries allowed: numpy, pandas, scikit-learn, matplotlib (optional)
# Saves & loads ONLY from Google Drive; no local runtime writes.
# ============================

import os, re, gc, json, math, string, warnings
warnings.filterwarnings("ignore")

# low-level math threads (used by BLAS/OpenMP inside sklearn/numpy)
MATH_THREADS = 8            # good starting point for 20 cores; bump to 10–12 if stable

os.environ["OMP_NUM_THREADS"]      = str(MATH_THREADS)
os.environ["MKL_NUM_THREADS"]      = str(MATH_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(MATH_THREADS)
os.environ["NUMEXPR_NUM_THREADS"]  = str(MATH_THREADS)

# Optional: keep threads hot & pinned
os.environ["KMP_AFFINITY"]  = "granularity=fine,compact,1,0"
os.environ["KMP_BLOCKTIME"] = "0"   # reduce thread sleep lag

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.feature_extraction.text import (
    TfidfVectorizer, CountVectorizer, HashingVectorizer, TfidfTransformer
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from joblib import dump, load

In [3]:
# -------------------------------------------------------
# Config
# -------------------------------------------------------
RANDOM_STATE = 42
JOBS = 2                        # keep memory stable
BASE_DIR = "model_weights_kaggle_challenge_1"
os.makedirs(BASE_DIR, exist_ok=True)

In [4]:
# -------------------------------------------------------
# Data
# -------------------------------------------------------
train = pd.read_csv("train.csv")
val   = pd.read_csv("val.csv")
test  = pd.read_csv("test.csv")

assert {'id','text','label'}.issubset(train.columns)
assert {'id','text','label'}.issubset(val.columns)
assert {'id','text'}.issubset(test.columns)

X_train_text = train["text"].values
y_train      = train["label"].astype(int).values
X_val_text   = val["text"].values
y_val        = val["label"].astype(int).values
X_test_text  = test["text"].values

print("Train label distribution:\n",
      train['label'].value_counts(normalize=True).rename('proportion').round(3))
print("Val label distribution:\n",
      val['label'].value_counts(normalize=True).rename('proportion').round(3))

Train label distribution:
 label
0    0.708
1    0.292
Name: proportion, dtype: float64
Val label distribution:
 label
1    0.507
0    0.493
Name: proportion, dtype: float64


In [5]:
# -------------------------------------------------------
# Utility: stylistic features (dense float32)
# -------------------------------------------------------
class StylisticFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stopwords = set((
            "the","and","is","in","to","of","that","it","for","on","you","with","as","this",
            "are","be","or","by","an","from","at","was","have","not","but","we","they","which",
            "one","all","can","has","there","their","more","will","if","about","so","what"
        ))
        self.punct_set = set(string.punctuation)

    def _safe_div(self, a, b): return float(a)/float(b) if b else 0.0

    def _char_entropy(self, s):
        if not s: return 0.0
        counts = {}
        for ch in s: counts[ch] = counts.get(ch, 0) + 1
        n = len(s); ent = 0.0
        for c in counts.values():
            p = c/n
            ent -= p * math.log(p + 1e-12, 2)
        return ent

    def fit(self, X, y=None): return self

    def transform(self, X):
        out = []
        for t in X:
            s = t if isinstance(t, str) else ""
            s = s.strip()
            n_chars = len(s)
            words = re.findall(r"\b\w+\b", s.lower())
            n_words = len(words)
            unique = set(words)
            n_unique = len(unique)

            avg_word_len = np.mean([len(w) for w in words]) if n_words else 0.0
            long_word_ratio = self._safe_div(sum(1 for w in words if len(w) >= 7), n_words)

            punct_cnt = sum(1 for ch in s if ch in self.punct_set)
            digit_cnt = sum(1 for ch in s if ch.isdigit())
            upper_cnt = sum(1 for ch in s if ch.isupper())
            space_cnt = sum(1 for ch in s if ch.isspace())
            stop_cnt  = sum(1 for w in words if w in self.stopwords)

            hapax_ratio = 0.0
            if n_words:
                freq = {}
                for w in words: freq[w] = freq.get(w, 0) + 1
                hapax_ratio = self._safe_div(sum(1 for c in freq.values() if c == 1), n_words)

            feats = [
                n_chars, n_words, avg_word_len, self._safe_div(n_unique, n_words),
                hapax_ratio, long_word_ratio, self._safe_div(punct_cnt, n_chars),
                self._safe_div(digit_cnt, n_chars), self._safe_div(upper_cnt, n_chars),
                self._safe_div(space_cnt, n_chars), self._safe_div(stop_cnt, n_words),
                self._char_entropy(s)
            ]
            out.append(feats)
        return np.asarray(out, dtype=np.float32)

In [6]:
# -------------------------------------------------------
# Helpers
# -------------------------------------------------------
def gc_clear(*objs):
    for o in objs:
        try: del o
        except: pass
    gc.collect()

def best_saved_file(prefix):
    """Return path to saved file with highest F1 in name for given prefix, else None."""
    if not os.path.isdir(BASE_DIR): return None
    cand = []
    for fn in os.listdir(BASE_DIR):
        if fn.startswith(prefix) and fn.endswith(".joblib"):
            # Expect pattern ..._F1_<score>.joblib
            m = re.search(r"_F1_([0-9]*\.?[0-9]+)", fn)
            if m:
                cand.append((float(m.group(1)), fn))
    if not cand: return None
    cand.sort(reverse=True)
    return os.path.join(BASE_DIR, cand[0][1])

def save_artifact(prefix, hyper_s, f1, obj):
    fn = f"{prefix}_{hyper_s}_F1_{f1:.4f}.joblib"
    path = os.path.join(BASE_DIR, fn)
    dump(obj, path)
    return path

def tune_threshold(y_true, scores, lo=0.1, hi=0.9, num=101):
    thr, best = 0.5, -1.0
    for t in np.linspace(lo, hi, num):
        pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, pred)
        if f1 > best:
            thr, best = t, f1
    return thr, best

def rank_scores(vec):
    r = (vec.argsort().argsort()).astype(np.float32)
    return r / (len(r) - 1 + 1e-9)



In [7]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [8]:
# -------------------------------------------------------
# Model trainers (each returns dict with: model, feats, extra, thr, score_kind, val_f1, params_str)
# -------------------------------------------------------
def train_LR():
    word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=3, max_df=0.90,
                           max_features=40000, sublinear_tf=True, strip_accents='unicode',
                           lowercase=True, dtype=np.float32)
    char = TfidfVectorizer(analyzer='char', ngram_range=(3,4), min_df=3, max_df=1.0,
                           max_features=30000, sublinear_tf=True, lowercase=False, dtype=np.float32)
    num  = Pipeline([("stylo", StylisticFeatures()), ("scale", StandardScaler(with_mean=False))])
    feats = FeatureUnion([("word", word), ("char", char), ("num", num)], n_jobs=JOBS)

    Xtr = feats.fit_transform(X_train_text)
    Xva = feats.transform(X_val_text)

    grid = GridSearchCV(
        LogisticRegression(solver="saga", max_iter=3000, random_state=RANDOM_STATE, n_jobs=JOBS),
        param_grid={"C":[1.0,2.0,4.0], "penalty":["l2"], "class_weight":[None,"balanced"]},
        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1
    )
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.2, 0.8, 61)
    params_s = f"C{clf.C}_l2_{'bal' if clf.class_weight=='balanced' else 'none'}"
    gc_clear(Xtr, Xva, grid)
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [9]:
def train_SVC():
    word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=3, max_df=0.90,
                           max_features=35000, sublinear_tf=True, strip_accents='unicode',
                           lowercase=True, dtype=np.float32)
    char = TfidfVectorizer(analyzer='char', ngram_range=(3,5), min_df=3, max_df=1.0,
                           max_features=25000, sublinear_tf=True, lowercase=False, dtype=np.float32)
    num  = Pipeline([("stylo", StylisticFeatures()), ("scale", StandardScaler(with_mean=False))])
    feats = FeatureUnion([("word", word), ("char", char), ("num", num)], n_jobs=JOBS)

    Xtr = feats.fit_transform(X_train_text)
    Xva = feats.transform(X_val_text)

    grid = GridSearchCV(
        LinearSVC(random_state=RANDOM_STATE, max_iter=500),
        param_grid={"C":[0.5,1.0,2.0], "class_weight":[None,"balanced"], "loss":["hinge","squared_hinge"]},
        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1
    )
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    score = clf.decision_function(Xva)
    r = rank_scores(score)
    thr, f1 = tune_threshold(y_val, r, 0.2, 0.8, 61)
    params_s = f"C{clf.C}_{clf.loss}_{'bal' if clf.class_weight=='balanced' else 'none'}"
    gc_clear(Xtr, Xva, grid, score, r)
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "rank", "val_f1": f1, "params_str": params_s}



In [10]:
def train_CNB():
    word = CountVectorizer(analyzer='word', ngram_range=(1,2), min_df=3, max_df=0.90,
                           max_features=30000, strip_accents='unicode', lowercase=True, dtype=np.int32)
    char = CountVectorizer(analyzer='char', ngram_range=(3,4), min_df=3, max_df=1.0,
                           max_features=30000, lowercase=False, dtype=np.int32)
    num  = Pipeline([("stylo", StylisticFeatures()), ("scale", StandardScaler(with_mean=False))])
    feats = FeatureUnion([("word", word), ("char", char), ("num", num)], n_jobs=JOBS)

    Xtr = feats.fit_transform(X_train_text)
    Xva = feats.transform(X_val_text)

    grid = GridSearchCV(ComplementNB(), param_grid={"alpha":[0.5,1.0,2.0]},
                        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1)
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.1, 0.9, 81)
    params_s = f"a{clf.alpha}"
    gc_clear(Xtr, Xva, grid)
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [11]:
def train_MNB():
    word = CountVectorizer(analyzer='word', ngram_range=(1,2), min_df=3, max_df=0.90,
                           max_features=30000, strip_accents='unicode', lowercase=True, dtype=np.int32)
    char = CountVectorizer(analyzer='char', ngram_range=(3,4), min_df=3, max_df=1.0,
                           max_features=30000, lowercase=False, dtype=np.int32)
    feats = FeatureUnion([("word", word), ("char", char)], n_jobs=JOBS)

    Xtr = feats.fit_transform(X_train_text)
    Xva = feats.transform(X_val_text)

    grid = GridSearchCV(MultinomialNB(), param_grid={"alpha":[0.5,1.0,2.0]},
                        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1)
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.1, 0.9, 81)
    params_s = f"a{clf.alpha}"
    gc_clear(Xtr, Xva, grid)
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [12]:
def train_BNB():
    # binary counts directly from CountVectorizer(binary=True)
    word = CountVectorizer(analyzer='word', ngram_range=(1,2), min_df=3, max_df=0.90,
                           max_features=25000, strip_accents='unicode', lowercase=True,
                           binary=True, dtype=np.int32)
    char = CountVectorizer(analyzer='char', ngram_range=(3,5), min_df=3, max_df=1.0,
                           max_features=25000, lowercase=False, binary=True, dtype=np.int32)
    feats = FeatureUnion([("word", word), ("char", char)], n_jobs=JOBS)

    Xtr = feats.fit_transform(X_train_text)
    Xva = feats.transform(X_val_text)

    grid = GridSearchCV(BernoulliNB(), param_grid={"alpha":[0.5,1.0,2.0]},
                        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1)
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.1, 0.9, 81)
    params_s = f"a{clf.alpha}"
    gc_clear(Xtr, Xva, grid)
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [13]:
def train_SGD():
    # Hashing + TF-IDF keeps memory flat
    h_word = HashingVectorizer(analyzer='word', n_features=2**18, alternate_sign=False,
                               ngram_range=(1,2), strip_accents='unicode', lowercase=True,
                               norm=None, dtype=np.float32)
    h_char = HashingVectorizer(analyzer='char', n_features=2**18, alternate_sign=False,
                               ngram_range=(3,4), lowercase=False, norm=None, dtype=np.float32)
    tfidf  = TfidfTransformer(sublinear_tf=True)
    num    = Pipeline([("stylo", StylisticFeatures()), ("scale", StandardScaler(with_mean=False))])
    feats  = FeatureUnion([
        ("word", Pipeline([("hash", h_word), ("tfidf", tfidf)])),
        ("char", Pipeline([("hash", h_char), ("tfidf", tfidf)])),
        ("num",  num)
    ], n_jobs=JOBS)

    Xtr = feats.fit_transform(X_train_text)
    Xva = feats.transform(X_val_text)

    grid = GridSearchCV(
        SGDClassifier(loss="log_loss", penalty="l2", class_weight="balanced",
                      max_iter=1000, tol=1e-3, random_state=RANDOM_STATE),
        param_grid={"alpha":[1e-5, 3e-5, 1e-4]},
        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1
    )
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.2, 0.8, 61)
    params_s = f"a{clf.alpha}"
    gc_clear(Xtr, Xva, grid)
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [14]:
def train_KNN():
    # compact char TF-IDF + numeric; cosine
    char = TfidfVectorizer(analyzer='char', ngram_range=(3,5), min_df=3, max_df=1.0,
                           max_features=30000, sublinear_tf=True, lowercase=False, dtype=np.float32)
    num  = Pipeline([("stylo", StylisticFeatures()), ("scale", StandardScaler(with_mean=False))])
    feats = FeatureUnion([("char", char), ("num", num)], n_jobs=JOBS)

    Xtr = feats.fit_transform(X_train_text)
    Xva = feats.transform(X_val_text)

    norm = Normalizer(copy=False)
    Xtr = norm.fit_transform(Xtr)
    Xva = norm.transform(Xva)

    grid = GridSearchCV(
        KNeighborsClassifier(metric="cosine", algorithm="brute"),
        param_grid={"n_neighbors":[3,5,7]},
        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1
    )
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.2, 0.8, 61)
    params_s = f"k{clf.n_neighbors}"
    gc_clear(Xtr, Xva, grid)
    return {"model": clf, "feats": feats, "extra": {"norm": norm},
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [15]:
def train_RF():
    # Trees need low-dim dense inputs: use hashing -> tfidf -> SVD(200) + numeric
    h_word = HashingVectorizer(analyzer='word', n_features=2**18, alternate_sign=False,
                               ngram_range=(1,2), strip_accents='unicode', lowercase=True,
                               norm=None, dtype=np.float32)
    h_char = HashingVectorizer(analyzer='char', n_features=2**18, alternate_sign=False,
                               ngram_range=(3,4), lowercase=False, norm=None, dtype=np.float32)
    tfidf = TfidfTransformer(sublinear_tf=True)
    feats_text = FeatureUnion([
        ("word", Pipeline([("hash", h_word), ("tfidf", tfidf)])),
        ("char", Pipeline([("hash", h_char), ("tfidf", tfidf)]))
    ], n_jobs=JOBS)
    num = Pipeline([("stylo", StylisticFeatures())])  # raw numeric; trees don't need scaling

    # Build reduced dense train/val
    Xt_tr = feats_text.fit_transform(X_train_text)
    Xt_va = feats_text.transform(X_val_text)
    svd = TruncatedSVD(n_components=200, random_state=RANDOM_STATE)
    Ztr = svd.fit_transform(Xt_tr)
    Zva = svd.transform(Xt_va)

    Xtr = np.hstack([Ztr, StylisticFeatures().transform(X_train_text)])
    Xva = np.hstack([Zva, StylisticFeatures().transform(X_val_text)])

    grid = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=JOBS),
        param_grid={"n_estimators":[200], "max_depth":[12, None], "min_samples_leaf":[1,2]},
        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1
    )
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.2, 0.8, 61)
    params_s = f"n{clf.n_estimators}_d{clf.max_depth}_l{clf.min_samples_leaf}"
    gc_clear(Xt_tr, Xt_va, Ztr, Zva, Xtr, Xva, grid)
    # Save SVD + feats_text alongside
    feats = {"text": feats_text, "svd": svd, "num": StylisticFeatures()}
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [16]:
def train_ADA():
    # Same reduced setup as RF
    h_word = HashingVectorizer(analyzer='word', n_features=2**18, alternate_sign=False,
                               ngram_range=(1,2), strip_accents='unicode', lowercase=True,
                               norm=None, dtype=np.float32)
    h_char = HashingVectorizer(analyzer='char', n_features=2**18, alternate_sign=False,
                               ngram_range=(3,4), lowercase=False, norm=None, dtype=np.float32)
    tfidf = TfidfTransformer(sublinear_tf=True)
    feats_text = FeatureUnion([
        ("word", Pipeline([("hash", h_word), ("tfidf", tfidf)])),
        ("char", Pipeline([("hash", h_char), ("tfidf", tfidf)]))
    ], n_jobs=JOBS)
    num = StylisticFeatures()

    Xt_tr = feats_text.fit_transform(X_train_text)
    Xt_va = feats_text.transform(X_val_text)
    svd = TruncatedSVD(n_components=150, random_state=RANDOM_STATE)
    Ztr = svd.fit_transform(Xt_tr)
    Zva = svd.transform(Xt_va)

    Xtr = np.hstack([Ztr, num.transform(X_train_text)])
    Xva = np.hstack([Zva, num.transform(X_val_text)])

    grid = GridSearchCV(
        AdaBoostClassifier(random_state=RANDOM_STATE),
        param_grid={"n_estimators":[200, 300], "learning_rate":[0.5, 1.0]},
        scoring="f1", cv=cv5, n_jobs=JOBS, refit=True, verbose=1
    )
    grid.fit(Xtr, y_train)
    clf = grid.best_estimator_
    proba = clf.predict_proba(Xva)[:,1]
    thr, f1 = tune_threshold(y_val, proba, 0.2, 0.8, 61)
    params_s = f"n{clf.n_estimators}_lr{clf.learning_rate}"
    gc_clear(Xt_tr, Xt_va, Ztr, Zva, Xtr, Xva, grid)
    feats = {"text": feats_text, "svd": svd, "num": num}
    return {"model": clf, "feats": feats, "extra": None,
            "thr": thr, "score_kind": "proba", "val_f1": f1, "params_str": params_s}



In [ ]:
# -------------------------------------------------------
# Orchestrator: load-if-exists else train, then save with F1 in filename
# -------------------------------------------------------
TRAINERS = [
    ("BNB",  train_BNB),
    ("MNB",  train_MNB),
    ("CNB",  train_CNB),
    ("SGD",  train_SGD),
    ("LR",   train_LR),
    ("SVC",  train_SVC),
    ("KNN",  train_KNN),
    ("ADA",  train_ADA),
    ("RF",   train_RF),
]

ARTIFACTS = {}  # name -> dict

for name, fn in TRAINERS:
    print(f"\n=== {name}: load or train ===")
    existing = best_saved_file(name)
    if existing:
        print(f"Found saved model: {existing} — loading.")
        ARTIFACTS[name] = load(existing)
        continue
    # Train fresh
    art = fn()
    # Pack everything to save as one blob
    payload = {
        "model": art["model"],
        "feats": art["feats"],          # can be FeatureUnion or dict for RF/ADA
        "extra": art["extra"],          # e.g., Normalizer for KNN
        "thr":   float(art["thr"]),
        "score_kind": art["score_kind"],# "proba" or "rank"
        "val_f1": float(art["val_f1"]),
        "params_str": art["params_str"]
    }
    fn_path = save_artifact(name, art["params_str"], art["val_f1"], payload)
    print(f"Saved: {fn_path}")
    ARTIFACTS[name] = payload
    gc_clear(art, payload)




=== BNB: load or train ===
Found saved model: /content/drive/MyDrive/Colab Notebooks/model_weights_kaggle_challenge_1/BNB_a2.0_F1_0.5968.joblib — loading.

=== MNB: load or train ===
Found saved model: /content/drive/MyDrive/Colab Notebooks/model_weights_kaggle_challenge_1/MNB_a1.0_F1_0.5273.joblib — loading.

=== CNB: load or train ===
Found saved model: /content/drive/MyDrive/Colab Notebooks/model_weights_kaggle_challenge_1/CNB_a0.5_F1_0.5543.joblib — loading.

=== SGD: load or train ===
Found saved model: /content/drive/MyDrive/Colab Notebooks/model_weights_kaggle_challenge_1/SGD_a1e-05_F1_0.7968.joblib — loading.

=== LR: load or train ===
Fitting 5 folds for each of 6 candidates, totalling 30 fits


In [ ]:
# -------------------------------------------------------
# Validation-time ensemble (9 models, hard majority)
# -------------------------------------------------------
print("\n=== Ensemble on validation (hard vote, 9 models) ===")
ind_preds = []
for name in [n for n,_ in TRAINERS]:
    art = ARTIFACTS[name]
    score_kind = art["score_kind"]
    thr = art["thr"]

    # Build validation features and scores
    if name in ("RF", "ADA"):
        feats_text = art["feats"]["text"]; svd = art["feats"]["svd"]; num = art["feats"]["num"]
        Xt = feats_text.transform(X_val_text)
        Z = svd.transform(Xt)
        Xva = np.hstack([Z, num.transform(X_val_text)])
        proba = art["model"].predict_proba(Xva)[:,1]
        pred = (proba >= thr).astype(int)
        gc_clear(Xt, Z, Xva, proba)

    else:
        feats = art["feats"]
        Xv = feats.transform(X_val_text)
        if name == "KNN" and art["extra"] and "norm" in art["extra"]:
            Xv = art["extra"]["norm"].transform(Xv)

        if score_kind == "proba":
            proba = art["model"].predict_proba(Xv)[:,1]
            pred = (proba >= thr).astype(int)
        else:
            # rank-based threshold for SVC
            s = art["model"].decision_function(Xv)
            r = rank_scores(s)
            pred = (r >= thr).astype(int)
        gc_clear(Xv)

    ind_preds.append(pred)

stack = np.vstack(ind_preds)
vote_val = (stack.sum(axis=0) >= 5).astype(int)   # majority of 9
f1_ens = f1_score(y_val, vote_val)
print(f"\nENSEMBLE VAL F1 = {f1_ens:.4f}")
print(classification_report(y_val, vote_val, digits=4))



In [ ]:
# -------------------------------------------------------
# Final refit on train+val with best hyperparams, then predict test
# Saved as *_FINAL.joblib (hyperparams included; F1 uses validation score)
# -------------------------------------------------------
print("\n=== Refit each model on train+val and predict test ===")
final_preds = []

for name in [n for n,_ in TRAINERS]:
    art = ARTIFACTS[name]
    thr = art["thr"]
    score_kind = art["score_kind"]

    if name in ("RF", "ADA"):
        feats_text = art["feats"]["text"]; svd = art["feats"]["svd"]; num = art["feats"]["num"]

        Xt_trv = feats_text.fit_transform(np.concatenate([X_train_text, X_val_text]))
        Ztrv = svd.fit_transform(Xt_trv)
        Xtrv = np.hstack([Ztrv, num.transform(np.concatenate([X_train_text, X_val_text]))])

        y_trv = np.concatenate([y_train, y_val])

        if name == "RF":
            params = art["model"].get_params()
            clf = RandomForestClassifier(**{k: params[k] for k in ["n_estimators","max_depth","min_samples_leaf","random_state","n_jobs"]})
        else:
            params = art["model"].get_params()
            clf = AdaBoostClassifier(**{k: params[k] for k in ["n_estimators","learning_rate","random_state"]})
        clf.fit(Xtrv, y_trv)

        # Save final
        payload = {"model": clf, "feats":{"text":feats_text,"svd":svd,"num":num},
                   "extra": None, "thr": thr, "score_kind": "proba",
                   "val_f1": art["val_f1"], "params_str": art["params_str"]}
        final_path = os.path.join(BASE_DIR, f"{name}_{art['params_str']}_F1_{art['val_f1']:.4f}_FINAL.joblib")
        dump(payload, final_path)

        # Predict test
        Xt_te = feats_text.transform(X_test_text)
        Zte = svd.transform(Xt_te)
        Xte = np.hstack([Zte, num.transform(X_test_text)])
        proba = clf.predict_proba(Xte)[:,1]
        pred = (proba >= thr).astype(int)
        final_preds.append(pred)
        gc_clear(Xt_trv, Ztrv, Xtrv, y_trv, Xt_te, Zte, Xte, proba)

    else:
        feats = art["feats"]
        # Refit featurizer on train+val, then refit model with same params
        Xtrv_text = np.concatenate([X_train_text, X_val_text])
        y_trv = np.concatenate([y_train, y_val])

        Xtrv = feats.fit_transform(Xtrv_text)
        params = art["model"].get_params()
        # Rebuild model class to avoid accidental state carryover
        if name == "LR":
            clf = LogisticRegression(**{k: params[k] for k in ["C","penalty","class_weight","solver","max_iter","random_state","n_jobs"]})
        elif name == "SVC":
            clf = LinearSVC(**{k: params[k] for k in ["C","class_weight","loss","random_state","max_iter"]})
        elif name == "CNB":
            clf = ComplementNB(**{k: params[k] for k in ["alpha"]})
        elif name == "MNB":
            clf = MultinomialNB(**{k: params[k] for k in ["alpha"]})
        elif name == "BNB":
            clf = BernoulliNB(**{k: params[k] for k in ["alpha"]})
        elif name == "SGD":
            clf = SGDClassifier(**{k: params[k] for k in ["alpha","loss","penalty","class_weight","max_iter","tol","random_state"]})
        elif name == "KNN":
            clf = KNeighborsClassifier(**{k: params[k] for k in ["n_neighbors","metric","algorithm"]})
        else:
            raise ValueError("Unknown model")

        # Extra normalizer for KNN
        extra = art["extra"]
        if name == "KNN":
            norm = Normalizer(copy=False)
            Xtrv = norm.fit_transform(Xtrv)
            clf.fit(Xtrv, y_trv)
            payload = {"model": clf, "feats": feats, "extra": {"norm": norm},
                       "thr": thr, "score_kind": art["score_kind"],
                       "val_f1": art["val_f1"], "params_str": art["params_str"]}
            final_path = os.path.join(BASE_DIR, f"{name}_{art['params_str']}_F1_{art['val_f1']:.4f}_FINAL.joblib")
            dump(payload, final_path)

            Xte = feats.transform(X_test_text); Xte = norm.transform(Xte)
            proba = clf.predict_proba(Xte)[:,1]
            pred = (proba >= thr).astype(int)
            final_preds.append(pred)
            gc_clear(Xtrv, y_trv, Xte, proba)

        else:
            clf.fit(Xtrv, y_trv)
            payload = {"model": clf, "feats": feats, "extra": None,
                       "thr": thr, "score_kind": art["score_kind"],
                       "val_f1": art["val_f1"], "params_str": art["params_str"]}
            final_path = os.path.join(BASE_DIR, f"{name}_{art['params_str']}_F1_{art['val_f1']:.4f}_FINAL.joblib")
            dump(payload, final_path)

            Xte = feats.transform(X_test_text)
            if art["score_kind"] == "proba":
                proba = clf.predict_proba(Xte)[:,1]
                pred = (proba >= thr).astype(int)
            else:
                s = clf.decision_function(Xte)
                r = rank_scores(s)
                pred = (r >= thr).astype(int)
            final_preds.append(pred)
            gc_clear(Xtrv, y_trv, Xte)



In [ ]:
# -------------------------------------------------------
# Final hard vote on TEST (9 models)
# -------------------------------------------------------
stack_te = np.vstack(final_preds)
final_test_pred = (stack_te.sum(axis=0) >= 5).astype(int)

submission = pd.DataFrame({"id": test["id"], "label": final_test_pred})
out_csv = "submission_ensemble9.csv"
submission.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)
print("Done.")
